# 05 · Ensemble & Sample-Path Forecasts

Represent the predictive distribution by **draws**: $M$ simulated future trajectories
$\mathbf x^{(m)}=(x^{(m)}_{T+1},\dots,x^{(m)}_{T+H})$, $m=1..M$.

## Theory

### Why samples?
* **Universal**: any model that can *simulate* (state-space, bootstrap, Bayesian MCMC, DeepAR, diffusion models,
  foundation models like Chronos / TimesFM / Lag-Llama) produces samples.
* **Paths keep the dependence across horizons.** Marginal quantiles do not. With paths you can forecast any
  **functional**: total demand over a quarter, peak month, probability of ever exceeding capacity, time until stock runs out.
  For these, $\text{quantile}(\sum_h Y_h)\ne\sum_h\text{quantile}(Y_h)$.
* Monte-Carlo error of a sample estimate shrinks like $1/\sqrt M$.

### Kinds of ensembles
| Kind | Spread comes from | Example |
|---|---|---|
| Stochastic simulation | future noise (aleatoric) | `res.simulate(...)` from ARIMA/ETS |
| Bootstrap | re-sampled residuals | nb 03 |
| Parameter-uncertainty | draws of $\theta$ (epistemic) | $\theta^{(m)}\sim N(\hat\theta,\widehat{\text{Cov}})$, or posterior draws (nb 11) |
| Multi-model | model disagreement | ARIMA + ETS + naive |
| Initial-condition perturbation | uncertain starting state | weather NWP ensembles |
| Deep generative | learned distribution | DeepAR, TimeGrad, Chronos |

### Combining probabilistic forecasts
* **Linear pool (mixture)**: $F=\sum_i w_iF_i$. Pool the *samples*. Spread includes the models' **disagreement**, so the result
  is wider. If every member is calibrated, the pool is **over-dispersed** (Gneiting & Ranjan 2013).
* **Quantile averaging (Vincentization)**: $F^{-1}(\tau)=\sum_i w_iF_i^{-1}(\tau)$. Average the *quantiles*. Keeps
  shape and is narrower. Often better when members are similar but shifted.

### Verifying ensembles
* **Rank (Talagrand) histogram**: rank of the observation among the $M$ members. Uniform means calibrated (the
  ensemble analogue of PIT). U-shape means under-dispersed.
* **Spread–skill**: for a calibrated ensemble, mean ensemble std ≈ RMSE of the ensemble mean (× $\sqrt{(M+1)/M}$).
* **Sample CRPS** $=\frac1M\sum|x_m-y|-\frac1{2M^2}\sum_{m,m'}|x_m-x_{m'}|$. Energy score for whole paths (nb 09).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(5)

air = pf.load_air_passengers()
train, test = air[:"1958-12-01"], air["1959-01-01":]
H, M = len(test), 1000

## 1. Four ways to generate sample paths (origin Dec-1958, 24 months ahead)

In [2]:
ORDER, SORDER = (0, 1, 1), (0, 1, 1, 12)

def sarima_paths(series, H, M):
    mod = SARIMAX(np.log(series), order=ORDER, seasonal_order=SORDER)
    res = mod.fit(disp=False)
    return np.exp(np.asarray(res.simulate(H, repetitions=M, anchor="end")).reshape(H, M).T), mod, res

def ets_paths(series, H, M):
    r = ETSModel(series, error="mul", trend="add", seasonal="mul", seasonal_periods=12).fit(disp=False)
    return np.asarray(r.simulate(H, repetitions=M, anchor="end")).reshape(H, M).T

def snaive_boot_paths(series, H, M, rng):
    z = np.log(series.to_numpy())
    d = z[12:] - z[:-12]; drift = d.mean(); pool = d - drift
    zz = np.tile(np.r_[z, np.zeros(H)], (M, 1)); n = len(z)
    for t in range(n, n + H):
        zz[:, t] = zz[:, t - 12] + drift + rng.choice(pool, M)
    return np.exp(zz[:, n:])

P = {}
P["SARIMA (noise only)"], mod, res = sarima_paths(train, H, M)
# parameter-uncertainty ensemble: 100 parameter draws x 10 paths each
draws = rng.multivariate_normal(res.params, res.cov_params(), size=100)
draws[:, -1] = np.abs(draws[:, -1])
P["SARIMA + param. uncertainty"] = np.vstack([
    np.exp(np.asarray(mod.filter(p).simulate(H, repetitions=10, anchor="end")).reshape(H, 10).T) for p in draws])
P["ETS(M,A,M)"] = ets_paths(train, H, M)
P["Seasonal-naive bootstrap"] = snaive_boot_paths(train, H, M, rng)
{k: v.shape for k, v in P.items()}

{'SARIMA (noise only)': (1000, 24),
 'SARIMA + param. uncertainty': (1000, 24),
 'ETS(M,A,M)': (1000, 24),
 'Seasonal-naive bootstrap': (1000, 24)}

In [3]:
fig = pf.subplots(2, 2, titles=[f"{n}: 40 paths + 90% band" for n in P], shared_yaxes=True, height=780)
for k, (name, p) in enumerate(P.items()):
    r, c = k // 2 + 1, k % 2 + 1
    first = k == 0
    fig.add_trace(go.Scatter(x=train.index[-36:], y=train[-36:], mode="lines", name="history", legendgroup="history",
                             showlegend=first, line=dict(color=pf.HISTORY, width=1.5)), row=r, col=c)
    pf.add_paths(fig, test.index, p, color=pf.SERIES[0], name="sample paths (40 of 1000)", row=r, col=c, showlegend=first)
    for qq, lab in zip(np.quantile(p, [0.05, 0.95], axis=0), ["P5", "P95"]):
        fig.add_trace(go.Scatter(x=test.index, y=qq, mode="lines", name="90% band (P5 / P95)", legendgroup="band",
                                 showlegend=first and lab == "P5", line=dict(color=pf.INK["primary"], width=1.5, dash="dash"),
                                 hovertemplate=lab + " %{y:,.0f}<extra></extra>"), row=r, col=c)
    fig.add_trace(go.Scatter(x=test.index, y=test, mode="markers", name="actual", legendgroup="actual", showlegend=first,
                             marker=dict(color=pf.ACTUAL, size=8, line=dict(color=pf.INK["surface"], width=2)),
                             hovertemplate="actual %{y:,.0f}<extra></extra>"), row=r, col=c)
fig.update_layout(hovermode="x unified")
fig.show()

## 2. Path functionals: questions only samples can answer
Business questions about 1959 (the first 12 months):
* What is the **annual total**, and its 90% interval?
* What is the probability that **any** month exceeds 520 thousand passengers?
* What is the distribution of the **peak** month value?

The wrong shortcut is to add up the monthly P5 and P95 bounds.

In [4]:
p = P["SARIMA (noise only)"][:, :12]
annual = p.sum(axis=1)
y59 = test[:12]
q_month = np.quantile(p, [0.05, 0.95], axis=0)
print(f"Annual total 1959  : 90% PI from paths  = [{np.quantile(annual,.05):.0f}, {np.quantile(annual,.95):.0f}]   (actual {y59.sum():.0f})")
print(f"                     WRONG: sum of monthly P5/P95 = [{q_month[0].sum():.0f}, {q_month[1].sum():.0f}]  -> too wide (implicitly assumes perfect correlation)")
print(f"P(any month > 520) = {np.mean((p > 520).any(axis=1)):.3f}     max over months of P(month > 520) = {np.max(np.mean(p > 520, axis=0)):.3f}")
print(f"Peak month: median {np.median(p.max(axis=1)):.0f}, 90% PI [{np.quantile(p.max(axis=1), .05):.0f}, {np.quantile(p.max(axis=1), .95):.0f}]  (actual {y59.max():.0f})")

# correlation of path values across horizons: why sums of quantiles fail
corr = np.corrcoef(p.T)
hh = np.arange(1, 13)
fig = go.Figure(go.Heatmap(z=corr, x=hh, y=hh, zmin=0, zmax=1, xgap=2, ygap=2,
                           colorscale=[[0, pf.BLUE[100]], [0.5, pf.BLUE[400]], [1, pf.BLUE[650]]],
                           colorbar=dict(title="corr"), hovertemplate="h=%{x} vs h=%{y}: corr %{z:.2f}<extra></extra>"))
fig.update_layout(title="Correlation between horizons in SARIMA paths", width=560, height=500,
                  xaxis=dict(title="h", showgrid=False, dtick=1), yaxis=dict(title="h", showgrid=False, dtick=1, autorange="reversed"))
fig.show()

Annual total 1959  : 90% PI from paths  = [4340, 5227]   (actual 5140)
                     WRONG: sum of monthly P5/P95 = [4265, 5328]  -> too wide (implicitly assumes perfect correlation)
P(any month > 520) = 0.463     max over months of P(month > 520) = 0.403
Peak month: median 517, 90% PI [457, 578]  (actual 559)


Summing monthly quantiles is only correct if the months are *perfectly* correlated (comonotonic). Here the months
are highly but not perfectly correlated, so the sum is somewhat too wide. Assuming independence would make it far
too narrow. Both errors get larger for longer horizons and less persistent series. Only paths give the right answer, because they carry the actual dependence
(the correlation matrix above).

## 3. Combining models: linear pool vs quantile averaging

In [5]:
taus = np.linspace(0.01, 0.99, 99)
members = ["SARIMA (noise only)", "ETS(M,A,M)", "Seasonal-naive bootstrap"]
yt = test.to_numpy()

def crps_from_q(qmat):          # qmat: (len(taus), H)
    return pf.crps_quantiles(yt, dict(zip(taus, qmat))).mean()

results = {name: crps_from_q(np.quantile(P[name], taus, axis=0)) for name in P}
pool = np.vstack([P[k] for k in members])                                    # mixture of samples
results["Linear pool (3 models)"] = crps_from_q(np.quantile(pool, taus, axis=0))
results["Quantile average (3 models)"] = crps_from_q(np.mean([np.quantile(P[k], taus, axis=0) for k in members], axis=0))

width = {name: np.mean(np.diff(np.quantile(P[name], [0.05, 0.95], axis=0), axis=0)) for name in P}
width["Linear pool (3 models)"] = np.mean(np.diff(np.quantile(pool, [0.05, 0.95], axis=0), axis=0))
width["Quantile average (3 models)"] = np.mean(np.mean([np.diff(np.quantile(P[k], [0.05, 0.95], axis=0), axis=0) for k in members], axis=0))
pd.DataFrame({"CRPS (1959-60)": results, "mean 90% width": width}).sort_values("CRPS (1959-60)")

,CRPS (1959-60),mean 90% width
Seasonal-naive bootstrap,11.767,120.720
Quantile average (3 models),17.819,124.256
Linear pool (3 models),18.582,146.598
SARIMA + param. uncertainty,23.206,140.987
SARIMA (noise only),24.294,135.535
"ETS(M,A,M)",33.451,116.515


**Caution:** this is a single forecast origin (24 correlated errors), so rankings here are noisy. In §4's rolling
evaluation over 60 origins, SARIMA beats the seasonal-naive bootstrap. Never pick a model from one origin.

The linear pool is always at least as wide as the average member (mixture variance = average variance + variance of
the means). Quantile averaging has exactly the average width. Which one wins depends on whether the members disagree
because of genuine uncertainty (pool) or because of bias (averaging).

## 4. Rolling verification: rank histograms and spread–skill
Origins every month from Jan-1955 to Dec-1959, horizon 1–12, 99 ensemble members per forecast.

In [6]:
rows = []
for o in range(72, len(air)):
    tr = air.iloc[:o]
    h_max = min(12, len(air) - o)
    ens = {"SARIMA": sarima_paths(tr, h_max, 99)[0], "Seasonal-naive bootstrap": snaive_boot_paths(tr, h_max, 99, rng)}
    obs = air.iloc[o:o + h_max].to_numpy()
    for name, e in ens.items():
        for h in range(h_max):
            rows.append(dict(model=name, origin=o, h=h + 1, y=obs[h], rank=int(np.sum(e[:, h] < obs[h])),
                             ens_mean=e[:, h].mean(), ens_std=e[:, h].std(ddof=1),
                             crps=float(pf.crps_sample(obs[h], e[:, h]))))
rv = pd.DataFrame(rows)
rv.groupby("model")[["crps"]].mean()

,crps
model,
SARIMA,10.805
Seasonal-naive bootstrap,12.204


In [7]:
MODELS = ["SARIMA", "Seasonal-naive bootstrap"]
fig = pf.subplots(1, 3, titles=[f"Rank histogram: {m}" for m in MODELS] + ["Spread–skill (solid = spread, dashed = RMSE)"],
                  height=420)
for k, mname in enumerate(MODELS):
    g = rv[rv.model == mname]
    pf.add_pit(fig, (g["rank"] + 0.5) / 100, bins=10, row=1, col=k + 1, color=pf.SERIES[k])   # rank scaled to (0, 1)
    fig.update_xaxes(title_text="rank of obs among 99 members (scaled)", row=1, col=k + 1)
    ss = g.groupby("h").apply(lambda d: pd.Series({"spread": d.ens_std.mean(), "rmse": np.sqrt(np.mean((d.y - d.ens_mean) ** 2))}))
    fig.add_trace(go.Scatter(x=ss.index, y=ss.spread, mode="lines+markers", name=f"{mname}: ensemble spread",
                             line=dict(color=pf.SERIES[k], width=2), marker=dict(size=8)), row=1, col=3)
    fig.add_trace(go.Scatter(x=ss.index, y=ss.rmse, mode="lines+markers", name=f"{mname}: RMSE of ensemble mean",
                             line=dict(color=pf.SERIES[k], width=2, dash="dash"), marker=dict(size=8, symbol="x")), row=1, col=3)
fig.update_xaxes(title_text="h (months)", row=1, col=3)
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.3, x=0), margin=dict(b=150), height=470)
fig.show()

## 5. How many samples do I need?

In [8]:
p_full = P["SARIMA (noise only)"][:, 0]
ref = pf.crps_sample(yt[0], p_full)
for M_ in [10, 30, 100, 300, 1000]:
    est = [pf.crps_sample(yt[0], rng.choice(p_full, M_)) for _ in range(300)]
    print(f"M={M_:5d}: CRPS estimate mean={np.mean(est):6.2f}  sd={np.std(est):5.2f}   (reference {ref:.2f})")

M=   10: CRPS estimate mean=  8.09  sd= 2.87   (reference 7.44)
M=   30: CRPS estimate mean=  7.56  sd= 1.73   (reference 7.44)
M=  100: CRPS estimate mean=  7.49  sd= 0.93   (reference 7.44)
M=  300: CRPS estimate mean=  7.49  sd= 0.52   (reference 7.44)
M= 1000: CRPS estimate mean=  7.47  sd= 0.27   (reference 7.44)


Small ensembles give **noisy and biased** CRPS: the $\frac{1}{2M^2}$ estimator is biased for small $M$, and
"fair CRPS" uses $\frac{1}{2M(M-1)}$ instead. Use ≥ 100–200 samples for stable quantiles in the tails.

## 6. Foundation models also give samples
Pre-trained time-series foundation models such as **Chronos** (Amazon), **TimesFM** (Google), **Moirai** (Salesforce) and
**Lag-Llama** generate sample paths zero-shot. The evaluation tools in this notebook apply unchanged. Example (needs `pip install chronos-forecasting torch`):

```python
from chronos import ChronosPipeline
import torch
pipe = ChronosPipeline.from_pretrained("amazon/chronos-t5-small")
samples = pipe.predict(torch.tensor(train.values), prediction_length=24, num_samples=200)[0].numpy()  # (200, 24)
pf.crps_sample(test.values, samples).mean()
```

### Exercises
1. Add the parameter-uncertainty ensemble to the rolling verification. Does the rank histogram get flatter?
2. Optimise linear-pool weights by minimising CRPS on 1955–1957 and test on 1958–1960.
3. Forecast the probability that cumulative passengers in H1-1960 exceed 2 750 thousand.